# Typed Dictionary of Big Step Semantics

$$\newcommand\eval{\mathbf{eval}}$$
$$\newcommand\semRule[3]{\begin{array}{c} #1 \\ \hline #2 \\ \end{array}\;(\text{#3}) }$$
$$\newcommand\typeOf{\mathbf{typeOf}}$$

### Type Environments

Just like we have environments $\sigma$ that map identifiers to values, we need to have type environments $\alpha$ that map identifiers to types.

Example: 

- Enviromnent : $\sigma: \{ "x" \mapsto 2, "y" \mapsto true, "f" \mapsto Closure("x", Plus("x", Const(10))) \}$.
- Type Environment: $\alpha: \{ "x" \mapsto \mathbf{num},\ "y" \mapsto \mathbf{bool},\ "f" \mapsto \mathbf{num} => \mathbf{num} \}$.

We say that an environment $\sigma$ is _compatible_ with a type environment $\alpha$ if and only if 
- $\mathbf{domain}(\sigma) \subseteq \mathbf{domain}(\alpha)$ - everything defined in $\sigma$ must be typed in $\alpha$.
- $\sigma(x)$ is a value of type $\alpha(x)$ for every $x \in \mathbf{domain}(\sigma)$. If $v$ is the value of $x$ in $\sigma$, the type in $\alpha$ must be the type of value $v$.

**Definition** An expression `e` in a program has a type `t`  under type environment $\tau$ if and only if 
- for any environment $\sigma$ that is _compatible_ with $\alpha$, 
- evaluating the expression $e$ under $\sigma$ must yield one of the following possibilities:
   - A value of type $t$, 
   - Nonterminating execution, or
   - A runtime error due to division by zero.
   
   
We will use a special type value __typeerror__ to signify that an expression is mistyped. In practice, our type checker will simply throw an exception under type errors.

<hr>

### Constants

$$\semRule{}{\typeOf(\texttt{Const(f)}, \alpha) = \mathbf{num}}{Const}$$

<hr>

### Identifiers

$$\semRule{x \in \mathbf{domain}(\alpha)}{ \typeOf(\texttt{Ident(x)}, \alpha)  = \alpha(x)}{ident-ok}$$
$$\semRule{x \not\in \mathbf{domain}(\alpha)}{ \typeOf(\texttt{Ident(x)}, \alpha)  = \mathbf{typeerror}}{ident-nok}$$

<hr>

### Arithmetic

$$\semRule{\typeOf(\texttt{e1}, \alpha) = \mathbf{num},\ \typeOf(\texttt{e2}, \alpha) = \mathbf{num}, T \in \{ Plus, Minus, Mult, Div \} }{\typeOf(\texttt{T(e1, e2)}, \alpha) = \mathbf{num}}{arith-ok} $$

Note that we do not check if `Div` leads to a division by zero in our type checking rule. This is a difference between the interpreter and type checker.

Short circuit semantics for type error: 

$$\semRule{{\typeOf(\texttt{e1}, \alpha) \not= \mathbf{num}},\ T \in \{ Plus, Minus, Mult, Div \} }{\typeOf(\texttt{T(e1, e2)}, \alpha) = \mathbf{typeerror}}{arith-nok-1} $$

Note that if `e1` leads to a result that is not a number, we simply shortcut.

$$\semRule{\typeOf(\texttt{e1}, \alpha) = \mathbf{num},\ {\typeOf(\texttt{e2}, \alpha) \not= \mathbf{num}},\ T \in \{ Plus, Minus, Mult, Div \} }{\typeOf(\texttt{T(e1, e2)}, \alpha) = \mathbf{typeerror}}{arith-nok-2} $$

<hr>

### Comparison Operators

$$\semRule{\typeOf(\texttt{e1}, \alpha) = \typeOf(\texttt{e2}, \alpha), \ T \in \{ Eq, Geq, Leq\} }{\typeOf(\texttt{T(e1, e2)}, \alpha) = \mathbf{bool}}{comp-ok} $$

$$\semRule{\typeOf(\texttt{e1}, \alpha) \not = \mathbf{num}, \ T \in \{ Eq, Geq, Leq\} }{\typeOf(\texttt{T(e1, e2)}, \alpha) = \mathbf{typeerror}}{comp-nok-1} $$

$$\semRule{\typeOf(\texttt{e1}, \alpha) = \mathbf{num}, \ \typeOf(\texttt{e2}, \alpha) \not = \mathbf{num}, \ T \in \{ Eq, Geq, Leq\} }{\typeOf(\texttt{T(e1, e2)}, \alpha) = \mathbf{typeerror}}{comp-nok-1} $$

<hr>

### Boolean Operators

$$\semRule{\typeOf(\texttt{e1}, \alpha) = \mathbf{bool},\ \typeOf(\texttt{e2}, \alpha) = \mathbf{bool}, T \in \{ And, Or\} }{\typeOf(\texttt{T(e1, e2)}, \alpha) = \mathbf{bool}}{bool-ok} $$

Note that unlike the rule for the interpreter, the type checker does not __short circuit__ boolean operators. This is because, we cannot know in advance if the value of `e1` is true for an `Or` or false for an `And`.

$$\semRule{\typeOf(\texttt{e1}, \alpha) \not= \mathbf{bool}, \ T \in \{ And, Or\}}{\typeOf(\texttt{T(e1, e2)}, \alpha) = \mathbf{typeerror}}{bool-nok-1}$$

$$\semRule{\typeOf(\texttt{e1}, \alpha) = \mathbf{bool}, \ \typeOf(\texttt{e2}, \alpha) \not= \mathbf{bool}, \ T \in \{ And, Or\}}{\typeOf(\texttt{T(e1, e2)}, \alpha) = \mathbf{typeerror}}{bool-nok-2}$$

<hr>

### If-Then-Else

$$\semRule{\typeOf(\texttt{condExpr}, \alpha) = \mathbf{bool},  \typeOf(\texttt{thenExpr}, \alpha) = \typeOf(\texttt{elseExpr}, \alpha) = t,\ t \not= \mathbf{typeerror} }{\typeOf(\texttt{If(condExpr, thenExpr, elseExpr)}, \alpha) = t}{ite-ok} $$

$$\semRule{{\typeOf(\texttt{condExpr}, \alpha) \not= \mathbf{bool}} }{\typeOf(\texttt{If(condExpr, thenExpr, elseExpr)}, \alpha) = \mathbf{typeerror} }{ite-nok-1} $$

$$\semRule{\typeOf(\texttt{condExpr}, \alpha) = \mathbf{bool}, \ {\typeOf(\texttt{thenExpr}, \alpha) \not= \typeOf(\texttt{elseExpr}, \alpha)}}{\typeOf(\texttt{If(condExpr, thenExpr, elseExpr)}, \alpha) = \mathbf{typeerror}}{ite-nok-2} $$

<hr>

### Let Binding

$$\semRule{\typeOf(\texttt{xExpr}, \alpha) = \texttt{xType}, \
\typeOf(\texttt{body}, \alpha[\texttt{x} \mapsto \texttt{xType}]) = t}{ \typeOf(\texttt{Let(x, xType, xExpr, body)}, \alpha) =  t}{let-ok}$$

The rule has couple of things to note:
- Note how we check that `xExpr` must have type `xType`, the same type as the annotation for `x`.
- When type checking `body`, we add the mapping $x \mapsto \texttt{xType}$ to the type environment.

A type error results when `xExpr` does not match the type annotation for `x` in the binding.

$$\semRule{{\typeOf(\texttt{xExpr}, \alpha) \not= \texttt{xType}}}{ \typeOf(\texttt{Let(x, xType, xExpr, body)}, \alpha) = \mathbf{typeerror}}{let-nok-1}$$

Similarly, a type error results when `body` leads to a type error.

$$\semRule{\typeOf(\texttt{xExpr}, \alpha) = \texttt{xType}, \ {\typeOf(\texttt{body}, \alpha[x \mapsto \texttt{xType}]) = \mathbf{typeerror}}}{ \typeOf(\texttt{Let(x, xType, xExpr, body)}, \alpha) = \mathbf{typeerror} }{let-nok-2}$$

<hr>

### Function Definitions

$$\semRule{\typeOf(\texttt{body}, \alpha[\texttt{arg} \mapsto \texttt{argType}]) = \texttt{returnType},\ \texttt{returnType} \not= \mathbf{typeerror}}{
\typeOf(\texttt{FunDef(arg, argType, body)}) = (\texttt{argType} => \texttt{returnType})}{fun-ok}$$

$$\semRule{{\typeOf(\texttt{body}, \alpha[\texttt{arg} \mapsto \texttt{argType}]) =  \mathbf{typeerror}}}{
\typeOf(\texttt{FunDef(arg, argType, body)} = \mathbf{typeerror} }{fun-nok}$$

<hr>

### Function Calls

$$\semRule{\typeOf(\texttt{argExpr}, \alpha) = \texttt{t1}, \ \typeOf(\texttt{funcExpr}, \alpha) = \texttt{t1} => \texttt{t2}}{\typeOf(\texttt{FunCall(funcExpr, argExpr)}, \alpha) = \texttt{t2}}{funcall-ok}$$

Note that technically in our type system when we have a function type `t1 => t2`, we know for a fact
that `t1` and `t2` cannot be a type error. This makes it redundant to say `t1 != typeerror` or `t2 != typeerror` in the rule.

$$\semRule{\typeOf(\texttt{argExpr}, \alpha) = \texttt{t1}, \ {\typeOf(\texttt{funcExpr}, \alpha) \not= \texttt{t1} => \texttt{t2}}}{\typeOf(\texttt{FunCall(funcExpr, argExpr)}, \alpha) = \mathbf{typeerror}}{funcall-nok}$$

__Exercise__ Write rules when evaluating `funcExpr` or `argExpr` leads to a typeerror.

<hr>

### Recursion

$$\semRule{\typeOf(\texttt{funcBody}, \alpha[arg \mapsto \texttt{argType}, f \mapsto \texttt{argType => returnType}]) = \texttt{t2}\\
\typeOf(\texttt{letBody}, \alpha[f \mapsto \texttt{argType => returnType}]) = t}{
\typeOf(\texttt{LetRec(f, argType => returnType, arg, argType, funcBody, letBody)},\alpha) = t}{rec-ok}$$

The idea behind the rule is as follows. Take a let rec definition:

~~~
let f: argType => returnType = function (x: argType) funcBody
   in letBody
~~~

- We check that if `f` were typed as `argType => returnType` and `arg` were typed as `argType`, does `funcBody` have a type of `returnType`?
- If yes to the question above, then we evaluate the type of `letBody` with `f` now receiving the type `argType => returnType`.

<hr>

### References

$$\semRule{ \typeOf(\texttt{e}, \alpha) = t,\; t \not= \mathbf{typeerror} }{\typeOf(\texttt{NewRef(e)}, \alpha) = ref(t) }{newref-ok}$$

It says that if $\texttt{e}$ receives type $t$ under type environment $\alpha$ and it is not a type error, then $\texttt{NewRef(e)}$ must receive the type $ref(t)$ under $\alpha$.



(i) Complete the missing terms for the rule for `DeRef` OK rule.
$$\semRule{\typeOf(\texttt{e}, \alpha) = ref(t)}{\typeOf(\texttt{DeRef(e)}, \alpha) = \ t}{deref-ok}$$

(ii) Complete the missing terms for the rule for `DeRef` error.
$$\semRule{\typeOf(\texttt{e}, \alpha) = t,\ t \not= ref(t') }{\typeOf(\texttt{DeRef(e)}, \alpha) = \mathbf{typeerror}}{deref-nok}$$

(iii) Complete the missing terms for `AssignRef` OK rule.
$$\semRule{\typeOf(\texttt{e1}, \alpha) = ref(t), \color{none} \ \typeOf(\texttt{e2}, \alpha) = t}{\typeOf(\texttt{AssignRef(e1, e2)}, \alpha) = t}{assignref-ok}$$

<hr>

### Title



<hr>

### Title



<hr>

### Title



<hr>

### Title

